# verify08c: 既存ベクトル（ラベル付けCSV）→ 直接トリアージ ＋ F1評価

`評価用データ_label付け後.csv`（1行=1通報、列=各ノードのcode＝**ベクトルそのもの**）を入力に、
**BERT推論を行わず** `vector_triage.vector_to_triage` で直接トリアージする。列→BERTノードは `triage_pipeline.NODE_MAP` で対応。

**流れ**: ベクトルCSV → `NODE_MAP` で {bertノード:code} → `build_vector` → `vector_to_triage(vec, age)` → main(VE/SE/LE) → F1(ja_dataset_v3.xlsx)
- 年齢列がCSVに無いため、同id年齢を `train/df_validation_input_renamed.csv` から補完（無ければ age=None）
- verify08b（BERT→ベクトル）とは別ルート。**モデル・チェックポイント不要**。

In [ ]:
# ===== セットアップ（Colab対応・モデル不要）=====
import os, sys, glob, json, csv, subprocess
IN_COLAB = 'google.colab' in sys.modules
REPO_URL, REPO_BRANCH = 'https://github.com/enenen13/Emergency_task', 'feature/headache-ablation-notebook'
DRIVE_ROOT_OVERRIDE = ''   # Colabで dataset/ja_dataset_v3.xlsx 等を置くフォルダ（任意）

DRIVE_ROOT = None
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as _e:
        print('mount retry:', _e); drive.mount('/content/drive', force_remount=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                    'pyyaml', 'pandas', 'scikit-learn', 'openpyxl'], check=False)
    DRIVE_ROOT = DRIVE_ROOT_OVERRIDE or None
    # リポジトリ資産（モジュール/yaml/辞書）を取得（毎回 最新へ強制更新）
    if not os.path.isdir('Emergency_task'):
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=False)
    else:
        subprocess.run(['git', '-C', 'Emergency_task', 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=False)
        subprocess.run(['git', '-C', 'Emergency_task', 'reset', '--hard', 'FETCH_HEAD'], check=False)
    REPO_DIR = os.path.abspath('Emergency_task')
else:
    REPO_DIR = os.path.abspath('.')       # ローカルはカレント（プロジェクト直下）
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ★最新コードを確実に反映（ランタイム再起動 不要）:
#   git reset --hard で最新ファイルを取得済みでも、import済みモジュールはメモリに古いまま残る。
#   ここで捨てておく → 後続セルの import が最新ファイルを読み込む。
for _m in ('triage_pipeline', 'vector_triage', 'age_logic', 'feature_preprocess'):
    sys.modules.pop(_m, None)

MODE = 'vecinput'   # 出力ファイル名タグ（BERT非使用・ベクトル直接入力）

# --- ベクトルCSV（評価用データ_label付け後.csv）を探索。無ければ Colab アップロード ---
def _find_vec_csv():
    pats = []
    if DRIVE_ROOT:
        pats += [os.path.join(DRIVE_ROOT, '評価用データ_label付け後.csv')]
    pats += [os.path.join(REPO_DIR, '評価用データ_label付け後.csv'),
             '評価用データ_label付け後.csv',
             'dataset/評価用データ_label付け後.csv', 'train/評価用データ_label付け後.csv']
    for p in pats:
        h = sorted(glob.glob(p))
        if h:
            return h[0]
    return None
VEC_CSV = _find_vec_csv()
if VEC_CSV is None and IN_COLAB:
    from google.colab import files
    print('ベクトルCSV（評価用データ_label付け後.csv）をアップロードしてください…')
    _up = files.upload()
    VEC_CSV = next((n for n in _up if n.lower().endswith('.csv')), None)
assert VEC_CSV, 'ベクトルCSV(評価用データ_label付け後.csv)が見つかりません'
print('REPO_DIR =', REPO_DIR, '| VEC_CSV =', VEC_CSV)

In [ ]:
# ===== ベクトルCSV → {bertノード:code}（NODE_MAP経由）＋ 年齢補完 =====
import pandas as pd
import vector_triage as vt, triage_pipeline as tp
NODE_MAP, NODE_ORDER = tp.NODE_MAP, vt.NODE_ORDER

df = pd.read_csv(VEC_CSV, dtype=str).fillna('')
assert 'id' in df.columns, 'id列が必要です'
node_cols = [c for c in df.columns if c in NODE_MAP]         # 列名=yamlノードid（NODE_MAPキー）
skipped   = [c for c in df.columns if c not in NODE_MAP and c not in ('id', 'Conversation')]
print(f'ベクトルCSV {len(df)}行 / ノード列 {len(node_cols)}(NODE_MAP一致) / 未対応列 {len(skipped)}: {skipped[:8]}')

# 年齢の補完（任意）: 同idの年齢を別CSV(列: id, 年齢)から。無ければ age=None
AGE_MERGE_CSV = ''      # 明示指定するならパス。空なら下の候補を自動探索
def _load_age_map():
    cands = ([AGE_MERGE_CSV] if AGE_MERGE_CSV else []) + [
        os.path.join(REPO_DIR, 'train', 'df_validation_input_renamed.csv'),
        'train/df_validation_input_renamed.csv']
    if DRIVE_ROOT:
        cands.append(os.path.join(DRIVE_ROOT, 'df_validation_input_renamed.csv'))
    for p in cands:
        if p and os.path.exists(p):
            gg = pd.read_csv(p, dtype=str)
            if 'id' in gg.columns and '年齢' in gg.columns:
                return {str(i): v for i, v in zip(gg['id'], gg['年齢'])}, p
    return {}, None
age_map, age_src = _load_age_map()
print('年齢ソース:', age_src or '（無し→ age=None: ベクトルの年齢ノード予測をそのまま使用）')

def _to_code(v):
    v = str(v).strip()
    if v == '':
        return None
    try:
        return int(float(v))
    except Exception:
        return None

recs = []
for _, r in df.iterrows():
    bp = {}
    for c in node_cols:
        cd = _to_code(r[c])
        if cd is None:
            continue
        bp[NODE_MAP[c]] = cd        # 列(yamlノード) → bertノード
    recs.append({'id': str(r['id']), 'age': age_map.get(str(r['id'])), 'bert_pred': bp})
print('例: id=%s ノード数=%d age=%s' % (recs[0]['id'], len(recs[0]['bert_pred']), recs[0]['age']))

In [ ]:
# ===== ベクトル → トリアージ ＋ サマリ =====
from collections import Counter
results = []
for rec in recs:
    vec = vt.build_vector(rec['bert_pred'])
    res = vt.vector_to_triage(vec, age=rec['age'])
    results.append({'id': rec['id'], 'age': rec['age'], 'main': res.main, 'sub1': res.sub1,
                    'common_completed': res.common_completed, 'common_stop': res.common_stop,
                    'completed': ','.join(res.completed_symptoms or []),
                    'furthest_broke': res.furthest_broke_symptom, 'report': res.report()})

for r in results[:3]:
    print('=' * 54); print('id=%s age=%s' % (r['id'], r['age'])); print(r['report'])

print('\n==== サマリ ====')
print('main分布 :', dict(Counter(r['main'] for r in results)))
print('sub1分布 :', dict(Counter(r['sub1'] for r in results)))
cc = sum(1 for r in results if r['common_completed'])
print('共通フロー完了 : %d/%d' % (cc, len(results)))
print('共通が途切れた止まりノード上位 :',
      Counter(r['common_stop'] for r in results if not r['common_completed']).most_common(5))

## F1 / 混同行列（正解 = `ja_dataset_v3.xlsx` の Triage Label）

入力ベクトルの `id` を `Case_id` と突き合わせ、`Triage Label`(Very/Semi/Low → VE/SE/LE) を正解に `main` を評価する。
Colabでは実行時にxlsxをアップロード（`FORCE_UPLOAD_TRUTH=True`）。

In [ ]:
# ===== 集計6: F1 / 混同行列（正解=ja_dataset_v3.xlsx の Triage Label）=====
import os, glob, re
import pandas as pd

# --- 正解Excelを探索（repo dataset / DRIVE_ROOT / ローカル）---
def _find_truth_xlsx():
    pats = []
    if DRIVE_ROOT:
        pats += [os.path.join(DRIVE_ROOT, 'ja_dataset_v3.xlsx'),
                 os.path.join(DRIVE_ROOT, 'dataset', 'ja_dataset_v3.xlsx')]
    pats += [os.path.join(REPO_DIR, 'dataset', 'ja_dataset_v3.xlsx'),
             os.path.join(REPO_DIR, 'train', 'ja_dataset_v3.xlsx'),
             'dataset/ja_dataset_v3.xlsx', 'train/ja_dataset_v3.xlsx', 'ja_dataset_v3.xlsx']
    for p in pats:
        h = sorted(glob.glob(p))
        if h:
            return h[0]
    return None

# ★このセルで正解Excelをアップロードするモード
#   FORCE_UPLOAD_TRUTH=True    … 毎回このセルでアップロードを促す（Colab）
#   FORCE_UPLOAD_TRUTH=False   … まずパス探索し、無ければアップロードにフォールバック
FORCE_UPLOAD_TRUTH = True

TRUTH_XLSX = None if FORCE_UPLOAD_TRUTH else _find_truth_xlsx()
if TRUTH_XLSX is None:
    if IN_COLAB:
        from google.colab import files
        print('正解Excel（ja_dataset_v3.xlsx 等）をアップロードしてください…')
        _up = files.upload()   # ダイアログでxlsxを選択
        TRUTH_XLSX = next((n for n in _up if n.lower().endswith(('.xlsx', '.xls'))), None)
    else:
        # ローカル/非Colab: 強制アップロード指定でもパス探索にフォールバック
        TRUTH_XLSX = _find_truth_xlsx()
assert TRUTH_XLSX, ('正解Excelが指定されていません。Colabでは上のダイアログでxlsxを選択、'
                    'ローカルでは dataset/ か DRIVE_ROOT に ja_dataset_v3.xlsx を置いてください。')
print('TRUTH_XLSX =', TRUTH_XLSX)

# Triage Label(文字列) → main(VE/SE/LE) 正規化
def _norm_label(s):
    t = re.sub(r'[\s_\-]+', '', str(s)).lower()
    if t.startswith('very'):   return 'VE'   # very emergency
    if t.startswith('semi'):   return 'SE'   # semi-emergency
    if t.startswith('low'):    return 'LE'   # low-emergency
    return None

# シート 'ja' があればそれ、無ければ先頭シートを使用（アップロード版の差異に頑健化）
_xls = pd.ExcelFile(TRUTH_XLSX)
_sheet = 'ja' if 'ja' in _xls.sheet_names else _xls.sheet_names[0]
_df = _xls.parse(_sheet)
assert {'Case_id', 'Triage Label'} <= set(_df.columns),     f'必要列(Case_id, Triage Label)がありません。実際の列: {list(_df.columns)}'
truth = {}                       # id(str) -> 'VE'/'SE'/'LE'
skipped_labels = []
for _, r in _df.iterrows():
    lab = _norm_label(r['Triage Label'])
    if lab is None:
        skipped_labels.append(r['Triage Label']); continue
    truth[str(r['Case_id'])] = lab
print(f'正解 {len(truth)} 件を読み込み（label分布: '
      + str({v: sum(1 for x in truth.values() if x == v) for v in ["VE","SE","LE"]}) + '）')
if skipped_labels:
    print('  未知ラベルでスキップ:', dict(pd.Series(skipped_labels).value_counts()))

# --- 予測(results)と id で突き合わせ ---
pairs = [(str(r['id']), r['main']) for r in results if str(r['id']) in truth]
unmatched = [str(r['id']) for r in results if str(r['id']) not in truth]
y_true = [truth[i] for i, _ in pairs]
y_pred = [p for _, p in pairs]
print(f'\n突き合わせ {len(pairs)}/{len(results)} 件（正解に無い予測id {len(unmatched)}件）')
if unmatched[:10]:
    print('  正解に無いid例:', unmatched[:10])

# '複数完了' 等 VE/SE/LE以外は誤り扱い（そのまま渡すと confusion_matrix labels 外＝不一致になる）
_other = sum(1 for p in y_pred if p not in ('VE', 'SE', 'LE'))
if _other:
    print(f'  ※ main が VE/SE/LE 以外（複数完了 等）: {_other}件 → 誤りとして計上')

labels = ['VE', 'SE', 'LE']
try:
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
    print('\n===== F1 =====')
    print('macro-F1 :', round(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0), 4))
    print('micro-F1 :', round(f1_score(y_true, y_pred, labels=labels, average='micro', zero_division=0), 4))
    print('weighted-F1 :', round(f1_score(y_true, y_pred, labels=labels, average='weighted', zero_division=0), 4))
    print('\n', classification_report(y_true, y_pred, labels=labels, zero_division=0))
    print('混同行列 (行=正解, 列=予測)  順:', labels)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    hdr = '        ' + '  '.join(f'{l:>4}' for l in labels)
    print(hdr)
    for lab, row in zip(labels, cm):
        print(f'  {lab:>4} | ' + '  '.join(f'{v:>4}' for v in row.tolist()))
except ImportError:
    from collections import Counter
    print('sklearn無し → 簡易集計のみ')
    acc = sum(t == p for t, p in zip(y_true, y_pred)) / max(1, len(y_true))
    print('accuracy :', round(acc, 4))
    print('誤りの内訳(正解->予測):',
          dict(Counter(f'{t}->{p}' for t, p in zip(y_true, y_pred) if t != p)))

# --- 誤分類の一覧をCSV保存（原因追跡用）---
import csv
os.makedirs('output', exist_ok=True)
out_eval = f'output/validation_f1_eval_{MODE}.csv'
res_by_id = {str(r['id']): r for r in results}
with open(out_eval, 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.writer(f)
    w.writerow(['id', 'true_main', 'pred_main', 'correct', 'sub1', 'common_completed', 'common_stop', 'furthest_broke'])
    for i, p in pairs:
        r = res_by_id[i]
        w.writerow([i, truth[i], p, int(truth[i] == p), r['sub1'],
                    r['common_completed'], r['common_stop'], r['furthest_broke']])
print('\n評価CSV →', out_eval, f'({len(pairs)}行)')

In [ ]:
# ===== 出力: 1行1通報のトリアージ詳細CSV =====
import csv, os
os.makedirs('output', exist_ok=True)
out_detail = f'output/vecinput_triage_{MODE}.csv'
with open(out_detail, 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.writer(f)
    w.writerow(['id', 'age', 'main', 'sub1', 'common_completed', 'common_stop', 'completed', 'furthest_broke'])
    for r in results:
        w.writerow([r['id'], r['age'], r['main'], r['sub1'],
                    r['common_completed'], r['common_stop'], r['completed'], r['furthest_broke']])
print('詳細CSV →', out_detail, f'({len(results)}行)')

## 遷移トレース ＆ 到達度の可視化（追加分析）

`recs`（cell 2）と同じ回答・年齢で計算するので、上の **F1結果と整合**します（`common_completed` は cell 3 と一致）。

- **追加①** 各通報が共通フローのどのノードを通り、どこで止まった/症候別へ抜けたか（`route_to_protocol` で症候へ = 共通完了）。
- **追加②** 「共通 完了/途切れ × 症状 完了/途切れ」の **4象限**それぞれで、実際にどのトリアージになるかをサンプルで確認。
  - ①共通完了×症状完了 → **症状のトリアージが最優先**で採用。②③④ → 安全側で途切れVE。
- **追加③** 症状ごとに「どれくらい辿れたか（遷移数=歩数）」、共通を辿れた通報で症状が完了/途切れる **本数と割合**。

In [ ]:
# ===== 追加①: 共通フローの遷移トレース（各通報が共通のどこへ進み、どこで止まったか）=====
import pandas as pd
from collections import Counter
g = tp.load_graph()

def _answers(rec):
    """run_triage と同じ手順で回答を構築（年齢論理集合の充填まで含む）→ 経路がトリアージ結果と一致。"""
    ans = dict(vt.DEFAULT_BASE_ANSWERS)
    ans.update(tp.build_answers_from_bert(rec['bert_pred'], g, age=rec.get('age'), sex=rec.get('sex')))
    if rec.get('age') is not None and tp.age_logic is not None:
        try:
            ans = tp.age_logic.augment_answers_with_age(ans, rec['age'], g.index)
        except Exception:
            pass
    return ans

def common_trace(rec):
    ans = _answers(rec)
    cw = tp.traverse_common(g, ans)
    completed = cw.stop.startswith('reached:') or cw.stop.startswith('route_to')
    return {'id': rec['id'], 'common_path': ' → '.join(cw.path),
            'last_node': cw.path[-1] if cw.path else '',
            'stop': cw.stop, 'route': cw.route or '',
            'common_completed': completed}

ctrace = [common_trace(r) for r in recs]
ctdf = pd.DataFrame(ctrace)

print('=== 共通フローが「どこで止まった/抜けたか」分布（last_node / stop）===')
dist = ctdf.groupby(['last_node', 'stop']).size().sort_values(ascending=False)
for (ln, st), n in dist.items():
    print(f'  {ln:<22} {st:<18} {n:>4} 件 ({n / len(ctdf) * 100:4.1f}%)')
print(f'\n共通完了: {int(ctdf.common_completed.sum())}/{len(ctdf)} '
      f'({ctdf.common_completed.mean() * 100:.1f}%)   途切れ: {int((~ctdf.common_completed).sum())}')

print('\n=== サンプル: 先頭8件が共通のどこを通ったか ===')
for row in ctrace[:8]:
    mark = '完了' if row['common_completed'] else '途切れ'
    tail = f"  → [route_to:{row['route']}]" if row['route'] else ''
    print(f"id={row['id']:>4} [{mark}] stop={row['stop']}")
    print(f"   {row['common_path']}{tail}")

ctdf.head(10)

In [ ]:
# ===== 追加②: 4象限（共通 完了/途切れ × 症状 完了/途切れ）→ トリアージ サンプル =====
from collections import defaultdict

def full_detail(rec):
    ans = _answers(rec)                                   # 追加①で定義
    cw = tp.traverse_common(g, ans)
    common_completed = cw.stop.startswith('reached:') or cw.stop.startswith('route_to')
    comp, broke = [], []
    for pid in g.protocol_ids:
        w = tp.traverse_symptom(g, pid, ans)
        if not any(n in ans for n in w.path):             # 回答が乗らない症状=関与なし → 除外
            continue
        if w.triage and not w.broke_off:
            comp.append({'symptom': pid, 'triage': w.triage, 'transitions': w.transitions})
        else:
            broke.append({'symptom': pid, 'stop': w.stop, 'transitions': w.transitions})
    res = vt.vector_to_triage(vt.build_vector(rec['bert_pred']), age=rec.get('age'),
                              sex=rec.get('sex'), graph=g)
    return {'id': rec['id'], 'common_completed': common_completed,
            'sym_completed': len(comp) > 0, 'completed': comp, 'broke': broke,
            'main': res.main, 'sub1': res.sub1, 'reason': res.reason}

details = [full_detail(r) for r in recs]

QNAME = {(True, True):  '① 共通完了 × 症状完了  → 症状のトリアージを採用(最優先)',
         (True, False): '② 共通完了 × 症状途切れ → 途切れVE(安全側)',
         (False, True): '③ 共通途切れ × 症状完了 → 途切れVE(安全側)',
         (False, False):'④ 共通途切れ × 症状途切れ → 途切れVE(安全側)'}
quad = defaultdict(list)
for d in details:
    quad[(d['common_completed'], d['sym_completed'])].append(d)

print('=== 4象限の件数・割合 ===')
for k in [(True, True), (True, False), (False, True), (False, False)]:
    n = len(quad[k]); print(f'  {QNAME[k]:<44} : {n:>4} 件 ({n / len(details) * 100:4.1f}%)')

print('\n=== 各象限のサンプル ===')
for k in [(True, True), (True, False), (False, True), (False, False)]:
    lst = quad[k]
    print(f'\n[{QNAME[k]}]  ({len(lst)}件)')
    if not lst:
        print('   該当なし'); continue
    d = lst[0]
    comp = ', '.join(f"{c['symptom']}:{c['triage']}({c['transitions']}歩)" for c in d['completed']) or 'なし'
    brk = ', '.join(f"{b['symptom']}({b['transitions']}歩)" for b in d['broke'][:5]) or 'なし'
    print(f"   例 id={d['id']}  →  main={d['main']} / sub1={d['sub1']}")
    print(f"      完了症状 : {comp}")
    print(f"      途切れ症状: {brk}")
    print(f"      根拠     : {d['reason']}")

In [ ]:
# ===== 追加③: 症状の到達度（どれくらいたどれたか）＋ 共通完了時の 進む/途切れ 本数・割合 =====
import pandas as pd
rows = []
for d in details:                        # details は「追加②」で作成済み
    for c in d['completed']:
        rows.append({'id': d['id'], 'symptom': c['symptom'], 'status': '完了',
                     'transitions': c['transitions'], 'common_completed': d['common_completed']})
    for b in d['broke']:
        rows.append({'id': d['id'], 'symptom': b['symptom'], 'status': '途切れ',
                     'transitions': b['transitions'], 'common_completed': d['common_completed']})
sdf = pd.DataFrame(rows)

n_all = len(sdf)
print(f'=== 関与した症状レコード（回答が乗った症状 × 通報）: {n_all} 本 ===')
for s in ['完了', '途切れ']:
    n = int((sdf.status == s).sum())
    print(f'  {s}: {n} 本 ({n / n_all * 100:.1f}%)')

print('\n=== 「共通をたどれた(共通完了)」通報に限定：症状はどこまで進むか ===')
cc = sdf[sdf.common_completed]
if len(cc):
    for s in ['完了', '途切れ']:
        n = int((cc.status == s).sum())
        print(f'  {s}: {n} 本 / {len(cc)} 本中 ({n / len(cc) * 100:.1f}%)')
    print('\n  到達度(遷移数=どれだけ深く辿れたか) の統計:')
    stat = cc.groupby('status')['transitions'].agg(['count', 'mean', 'median', 'max']).round(2)
    print(stat.to_string())

print('\n=== 症状別: 完了/途切れ 本数と完了率（本数降順）===')
piv = sdf.pivot_table(index='symptom', columns='status', values='id', aggfunc='count', fill_value=0)
for c in ['完了', '途切れ']:
    if c not in piv.columns:
        piv[c] = 0
piv['計'] = piv['完了'] + piv['途切れ']
piv['完了率%'] = (piv['完了'] / piv['計'] * 100).round(1)
piv = piv.sort_values('計', ascending=False)[['完了', '途切れ', '計', '完了率%']]
print(piv.to_string())